# LLM-vetted multi-anchor theme entries

A step-by-step walkthrough of how a themed puzzle gets its theme words. The pipeline turns a theme into **several** genuinely on-theme grid entries in four steps:

1. **Candidate pool** &mdash; score every database word against the theme (cosine) and take the top few (all real crossword answers).
2. **LLM vetting** &mdash; one LLM call keeps only the words a solver would truly recognise as on-theme (and, optionally, may suggest its own), guarded by a two-tier validation.
3. **Placement** &mdash; pin the vetted anchors into the grid at agreeing intersections.
4. **Fill with fallback** &mdash; fill the rest of the grid around the anchors, trying anchor combinations so as many as possible land.

This is a code walkthrough: it peeks at the internal methods so the backend algorithm and the LLM call are visible under the hood.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys, logging, random
import pandas as pd

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from dotenv import load_dotenv; load_dotenv() # OPENAI_API_KEY: theme embedding + anchor-vetting call

logging.disable(logging.INFO) # keep walkthrough output clean (hide INFO / HTTP logs)

In [ ]:
from src.gridgpt.word_database_manager import WordDatabaseManager
from src.gridgpt.template_manager import select_template
from src.gridgpt.theme_manager import ThemeManager
from src.gridgpt.theme_anchor import ThemeAnchorSelector
from src.gridgpt.crossword_generator import CrosswordGenerator, generate_themed_crossword
from src.gridgpt.utils import load_parameters

word_db = WordDatabaseManager()
cfg = load_parameters()['theme_anchors'] # max_anchors, candidate_pool, min/max_chars, allow_llm_words, min_zipf
theme = 'food'
template = select_template(template_id='5x5_diagonal_cut')
cfg

## 1. The candidate pool (cosine shortlist)

`ThemeManager.get_anchor_candidates` scores every database word against the theme and returns the top `candidate_pool` words across all usable slot lengths (3-5), best-first by cosine similarity. Every one is a real, published crossword answer and serves as the raw material the LLM will vet.

In [ ]:
theme_manager = ThemeManager(theme, word_db)
candidates = theme_manager.get_anchor_candidates(
    pool_size=cfg['candidate_pool'], min_chars=cfg['min_chars'], max_chars=cfg['max_chars'],
)
print(f'{len(candidates)} candidates for theme {theme!r} (best-first):')
candidates

## 2. The LLM vetting call (what the LLM is for)

Cosine is noisy: it ranks some loosely-related words highly (for "food" it surfaces FUN). One LLM call re-ranks the candidates down to the ones a solver would truly recognise as on-theme. `ThemeAnchorSelector.select_anchors` runs that call, then validates the result.

In [ ]:
selector = ThemeAnchorSelector()
print('LLM connected:', selector.llm_connection_success)

anchors = selector.select_anchors(
    theme, candidates, word_db,
    max_anchors=cfg['max_anchors'], allow_llm_words=False,
    min_zipf=cfg['min_zipf'], min_chars=cfg['min_chars'], max_chars=cfg['max_chars'],
)
pd.DataFrame({'candidates (cosine)': pd.Series(candidates), 'vetted anchors (LLM)': pd.Series(anchors)})

### Under the hood: the actual LLM call

`select_anchors` delegates the model call to the internal `_request_anchor_words`. Here is exactly what gets sent (system + user prompt, built from `conf/base/prompts.yml`) and the raw ordered list the model returns, before any validation.

In [ ]:
# Rebuild the user prompt exactly as _request_anchor_words does, to see what the model reads.
own = selector.prompt['own_words_instructions_llm_not_allowed']
user_prompt = selector.prompt['user_prompt'].format(
    theme=theme,
    candidates='\n'.join(f'- {c}' for c in candidates),
    max_anchors=cfg['max_anchors'],
    own_words_instruction=own,
)
print('--- SYSTEM PROMPT ---')
print(selector.prompt['system_prompt'])
print('--- USER PROMPT ---')
print(user_prompt)

# The raw model output: an ordered word list, before the two-tier validation.
raw = selector._request_anchor_words(theme, candidates, cfg['max_anchors'], allow_llm_words=False)
print('--- RAW LLM WORDS ---')
print(raw)

## 3. Two-tier validation (the guardrails)

Whatever the model returns is validated before it can become a grid entry, because the LLM can drift off-list or hallucinate:

- **Tier 1 (both modes):** a word in the database passes immediately. This drops hallucinations even when own-words is off.
- **Tier 2 (own-words only):** a word *not* in the database passes only when `allow_llm_words` is on and it is letters-only, in the 3&ndash;5 length range, and common enough (`wordfreq` Zipf &ge; `min_zipf`).

In [ ]:
from wordfreq import zipf_frequency

checks = ['PLUTO', 'MADEUPWORD', 'TACO', 'ZZZQX']
pd.DataFrame([{
    'word': w,
    'in DB (Tier 1)': ThemeAnchorSelector._in_database(w, word_db.word_list_with_frequencies),
    'zipf': round(zipf_frequency(w, 'en'), 2),
    'valid own-word (Tier 2)': ThemeAnchorSelector._is_valid_own_word(
        w, cfg['min_zipf'], cfg['min_chars'], cfg['max_chars']),
} for w in checks])

### Switching own-word suggestions on vs off

With `allow_llm_words=False` (default) only database words survive. With it on, the model may also add real, common words that are not in the database (their clue is then LLM-generated).

In [ ]:
# Comparison on a broader theme
broad_theme = 'food'
tm2 = ThemeManager(broad_theme, word_db)
cand2 = tm2.get_anchor_candidates(
    pool_size=cfg['candidate_pool'], min_chars=cfg['min_chars'], max_chars=cfg['max_chars'])

def vet(allow):
    return selector.select_anchors(
        broad_theme, cand2, word_db, max_anchors=cfg['max_anchors'], allow_llm_words=allow,
        min_zipf=cfg['min_zipf'], min_chars=cfg['min_chars'], max_chars=cfg['max_chars'])

print(f'theme {broad_theme!r}')
print('  allow_llm_words=False:', vet(False))
print('  allow_llm_words=True :', vet(True))

In [ ]:
# Comparison on a sparser theme
sparse_theme = 'jazz'
tm3 = ThemeManager(sparse_theme, word_db)
cand3 = tm3.get_anchor_candidates(
    pool_size=cfg['candidate_pool'], min_chars=cfg['min_chars'], max_chars=cfg['max_chars'])

def vet(allow):
    return selector.select_anchors(
        sparse_theme, cand3, word_db, max_anchors=cfg['max_anchors'], allow_llm_words=allow,
        min_zipf=cfg['min_zipf'], min_chars=cfg['min_chars'], max_chars=cfg['max_chars'])

print(f'theme {sparse_theme!r}')
print('  allow_llm_words=False:', vet(False))
print('  allow_llm_words=True :', vet(True))

## 4. Placing multiple anchors (`place_theme_entries`)

`CrosswordGenerator.place_theme_entries` pins the anchors into distinct slots of matching length, checking that letters agree at every shared intersection. It is best-effort: an anchor that cannot be placed consistently is skipped.

In [ ]:
gen = CrosswordGenerator(word_db)

def show_grid(grid):
    return print(pd.DataFrame([[c if c != '#' else '' for c in row] for row in grid]))

random.seed(0)
working = gen.place_theme_entries(template, anchors)
print('placed anchors:', working['filled_slots'])
show_grid(working['grid'])

## 5. Fill around the anchors, with a combination fallback

Pinning several long words in a 5x5 often leaves a crossing that no database word can complete. So the generator tries anchor **combinations** (all N, then every N&minus;1 subset, ...), best-first, and keeps the first that both places fully and fills. This lands as many anchors as the grid can take, usually two.

In [ ]:
import itertools

def fill_rate(anchor_set, trials=20):
    ok = 0
    for s in range(trials):
        random.seed(s)
        w = gen.place_theme_entries(template, list(anchor_set))
        placed = w['filled_slots']
        if len(placed) == len(anchor_set) and gen.fill(template, seed_assignment=dict(placed)) is not None:
            ok += 1
    return f'{ok}/{trials}'

print('anchors:', anchors)
print(f'  fill rate with all {len(anchors)} pinned:', fill_rate(anchors))
for combo in itertools.combinations(anchors, 2):
    print(f'  fill rate with {combo}:', fill_rate(combo))

In [ ]:
# generate_themed_crossword runs that combination search internally and returns a finished grid.
random.seed(0)
crossword = generate_themed_crossword(template, theme_entries=anchors, word_db_manager=word_db)
print('anchors that landed:', sorted(crossword['seed_entries'].values()))
print('theme_entries:', {k: (w, round(t, 2)) for k, (w, t) in crossword['theme_entries'].items()})
show_grid(crossword['grid'])

## 6. The whole pipeline, as the backend runs it

This mirrors what the API route does for a themed request: one `ThemeManager` (shared theme embedding) scores the words and yields candidates, the `ThemeAnchorSelector` vets them, and `generate_themed_crossword` places and fills. The similarity map also biases the remaining slots on-theme.

In [ ]:
def build_themed(theme, template_id='5x5_diagonal_cut', allow_llm_words=False, seed=0):
    tm = ThemeManager(theme, word_db)
    sims = tm.score_all_words()
    cands = tm.get_anchor_candidates(
        pool_size=cfg['candidate_pool'], min_chars=cfg['min_chars'], max_chars=cfg['max_chars'])
    anchors = ThemeAnchorSelector().select_anchors(
        theme, cands, word_db, max_anchors=cfg['max_anchors'], allow_llm_words=allow_llm_words,
        min_zipf=cfg['min_zipf'], min_chars=cfg['min_chars'], max_chars=cfg['max_chars'])
    random.seed(seed)
    cw = generate_themed_crossword(
        select_template(template_id=template_id), theme_entries=anchors,
        theme_similarities=sims, word_db_manager=word_db)
    return anchors, cw

for th in ['planets', 'food', 'sports']:
    a, cw = build_themed(th)
    print(f'{th:9} anchors={a} -> landed={sorted(cw["seed_entries"].values())}')